# record Video using stored UV iterations

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

In [ ]:
import sys, os
sys.path.append('../')
import MeshFEM
import mesh, mesh_energy, viewer, benchmark
import numpy as np

In [ ]:
base_path = 'local_exps_with_uv/'

In [ ]:
model_name = 'cow2Disc'
model_path = f'../../../Models/SmallModels/{model_name}.off'
hessian_option = 'Always'
read_uv_path = os.path.join(base_path, model_name, hessian_option, 'UVs')

In [ ]:
video_folder = 'Videos'
video_dir = os.path.join(base_path, video_folder)
if not os.path.exists(video_dir):  os.makedirs(video_dir)

In [ ]:
def read_uv_data(directory, i):
    uv_fn = 'uv_ravel_iter_' + str(i) + '.npz'
    uv_data = np.load(os.path.join(directory, uv_fn))
    uv_min = uv_data['arr']
    return uv_min

In [ ]:
RECORD_VIDEO=True

In [ ]:
m = mesh.Mesh(model_path)

In [ ]:
uv = mesh_energy.NodalVars(m, 2)

In [ ]:
uv.setVars(read_uv_data(read_uv_path, 0))

In [ ]:
import mesh_operations

In [ ]:
numUVs = sum(1 for f in os.listdir(read_uv_path) if f.endswith(".npz"))

In [ ]:
uv0 = read_uv_data(read_uv_path, 0)
uv1 = read_uv_data(read_uv_path, numUVs - 1)
#uv_bbox = compute_bbox(np.vstack(( read_uv_data(read_uv_path, numUVs - 1))))

In [ ]:
m_union = mesh_operations.concatenateMeshes([(uv0.reshape(-1, 2), m.elements()), (uv1.reshape(-1, 2), m.elements())])

In [ ]:
v = viewer.Viewer(m_union, wireframe=True)
em = MeshFEM.EmbeddedMesh(m, uv)
v.update(mesh=em, preserveExisting=False)
v.makeOpaque(color='#48B3FF')
v.show()

In [ ]:
# v.antialiasedImage(renderScale=8, outputScale=2, lineWidthScale=0.25)

In [ ]:
def updateUV(directory):
    for i in range(1, numUVs):
        uv.setVars(read_uv_data(directory, i))
        v.update()

In [ ]:
video_fn = model_name + '_' + hessian_option + '_symmdsUVopt.mp4'
if RECORD_VIDEO: v.recordStart(os.path.join(video_dir, video_fn), renderScale=8, outputScale=2, framerate=3, lineWidthScale=0.25)
updateUV(read_uv_path)
if RECORD_VIDEO: v.recordStop()

In [ ]:
!open .

In [ ]:
import video_writer

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
# x = np.linspace(0, np.pi, 100)
# fig = plt.figure(figsize=(8, 4))
# plt.plot(x, np.sin(x));

# pw = video_writer.PlotVideoWriter('test_plot.mp4', plt.gcf(), dpi=300, )
# pw.writeFrame(plt.gcf())
# plt.close()
# for i in range(30):
#     fig = plt.figure(figsize=(8, 4))
#     plt.plot(x, np.sin(x + i / np.pi))
#     pw.writeFrame(plt.gcf())
#     plt.close()
# pw.finish()